In [24]:
import pandas as pd
import requests
import json
import tabulate
#!pip install openpyxl

In [25]:
XLSX_PATH = "data/INRA.xlsx"

df = pd.read_excel(XLSX_PATH)
print("Nombre de lignes :", len(df))
df.columns.tolist()

Nombre de lignes : 1133


['CODE',
 'LIBELLE',
 'TX_MS',
 'DMO',
 'UFL',
 'UFV',
 'MAT',
 'PDI',
 'BPR',
 'AMIDON',
 'DT6AMI',
 'DT6N',
 'NDF',
 'PDIA',
 'EE',
 'AGT',
 'MO',
 'EB',
 'FIRME',
 'NI_REF',
 'BAL_PRO_REF',
 'PF',
 'UEM',
 'VEC',
 'MM',
 'CB',
 'ADF',
 'ADL',
 'SUCRES',
 'MOF',
 'DMA',
 'DCB',
 'DNDF',
 'DADF',
 'DE',
 'ED',
 'EM',
 'DTN',
 'A_N',
 'B_N',
 'C_N',
 'DRN',
 'DT6MS',
 'DTMS',
 'A_MS',
 'B_MS',
 'C_MS',
 'DTAMI',
 'A_AMI',
 'B_AMI',
 'C_AMI',
 'P',
 'P_ABS',
 'P_CAR',
 'CA',
 'CA_ABS',
 'CA_CAR',
 'MG',
 'S',
 'NA',
 'CU',
 'ZN',
 'MN',
 'CO',
 'SE',
 'I',
 'K',
 'CL',
 'BACA',
 'BE',
 'VIT_A',
 'VIT_D',
 'VIT_E',
 'LYS_BP',
 'HIS_BP',
 'ARG_BP',
 'THR_BP',
 'VAL_BP',
 'MET_BP',
 'ILE_BP',
 'LEU_BP',
 'PHE_BP',
 'ASP_BP',
 'SER_BP',
 'GLU_BP',
 'PRO_BP',
 'GLY_BP',
 'ALA_BP',
 'TYR_BP',
 'CYS_TRP_BP',
 'LYS_DI',
 'HIS_DI',
 'ARG_DI',
 'THR_DI',
 'VAL_DI',
 'MET_DI',
 'ILE_DI',
 'LEU_DI',
 'PHE_DI',
 'ASP_DI',
 'SER_DI',
 'GLU_DI',
 'PRO_DI',
 'GLY_DI',
 'ALA_DI',
 'TYR_DI',
 'PAA']

In [28]:
column_mapping = {
    # Identifiants
    "CODE": "Code aliment INRA/AFZ [CODE]",
    "LIBELLE": "Nom de l'aliment (libellé officiel) [LIBELLE]",

    # Matière sèche, digestibilité
    "TX_MS": "Teneur en matière sèche (%) [TX_MS]",
    "DMO": "Digestibilité de la matière organique (%) [DMO]",

    # Protéines et azote
    "MAT": "Matières azotées totales (Protéines brutes, %) [MAT]",
    "PDI": "Protéines digestibles dans l’intestin [PDI]",
    "PDIA": "Protéines digestibles non dégradées dans le rumen [PDIA]",
    "BPR": "Besoins protéiques ruminants (ancienne dénomination) [BPR]",
    "BAL_PRO_REF": "Bilan protéique de référence [BAL_PRO_REF]",
    "PF": "Protéines fermentescibles [PF]",
    "DTN": "Digestibilité totale de l’azote (%) [DTN]",
    "A_N": "Fraction A de l’azote (soluble) [A_N]",
    "B_N": "Fraction B de l’azote (dégradable) [B_N]",
    "C_N": "Fraction C de l’azote (indégradable) [C_N]",
    "DRN": "Dégradabilité de l’azote [DRN]",

    # Amidon, sucres
    "AMIDON": "Teneur en amidon (%) [AMIDON]",
    "SUCRES": "Teneur en sucres (%) [SUCRES]",

    # Digestibilité amidon et MS
    "DT6AMI": "Digestibilité de l’amidon à 6h [DT6AMI]",
    "DT6N": "Digestibilité de l’azote à 6h [DT6N]",
    "DT6MS": "Digestibilité de la MS à 6h [DT6MS]",
    "DTMS": "Digestibilité de la matière sèche (%) [DTMS]",
    "DTAMI": "Digestibilité de l’amidon (%) [DTAMI]",
    "A_MS": "Fraction A de la matière sèche [A_MS]",
    "B_MS": "Fraction B de la matière sèche [B_MS]",
    "C_MS": "Fraction C de la matière sèche [C_MS]",
    "A_AMI": "Fraction A de l’amidon [A_AMI]",
    "B_AMI": "Fraction B de l’amidon [B_AMI]",
    "C_AMI": "Fraction C de l’amidon [C_AMI]",

    # Fibres
    "NDF": "Fibres au détergent neutre (NDF) [NDF]",
    "ADF": "Fibres au détergent acide (ADF) [ADF]",
    "ADL": "Lignine (ADL) [ADL]",
    "CB": "Cellulose brute [CB]",
    "MOF": "Matières organiques fermentescibles [MOF]",
    "DNDF": "Digestibilité NDF [DNDF]",
    "DADF": "Digestibilité ADF [DADF]",
    "DCB": "Digestibilité cellulose brute [DCB]",
    "DMA": "Digestibilité matière azotée [DMA]",
    "AGT": "Azote global total [AGT]",

    # Lipides / énergie
    "EE": "Extractif éthéré (lipides %) [EE]",
    "MO": "Matière organique (%) [MO]",
    "EB": "Énergie brute (MJ/kg) [EB]",
    "DE": "Énergie digestible (MJ/kg) [DE]",
    "ED": "Énergie digérée (MJ/kg) [ED]",
    "EM": "Énergie métabolisable (MJ/kg) [EM]",
    "UEM": "Unité Énergétique Monogastrique [UEM]",
    "VEC": "Valeur énergétique corrigée [VEC]",

    # Minéraux majeurs
    "MM": "Matière minérale (cendres %) [MM]",
    "P": "Phosphore total (%) [P]",
    "P_ABS": "Phosphore absorbé (%) [P_ABS]",
    "P_CAR": "Phosphore digestible (caractérisé) [P_CAR]",
    "CA": "Calcium total (%) [CA]",
    "CA_ABS": "Calcium absorbé (%) [CA_ABS]",
    "CA_CAR": "Calcium digestible (caractérisé) [CA_CAR]",
    "MG": "Magnésium (mg/kg) [MG]",
    "S": "Soufre (mg/kg) [S]",
    "NA": "Sodium (mg/kg) [NA]",
    "CU": "Cuivre (mg/kg) [CU]",
    "ZN": "Zinc (mg/kg) [ZN]",
    "MN": "Manganèse (mg/kg) [MN]",
    "CO": "Cobalt (mg/kg) [CO]",
    "SE": "Sélénium (mg/kg) [SE]",
    "I": "Iode (mg/kg) [I]",
    "K": "Potassium (mg/kg) [K]",
    "CL": "Chlore (mg/kg) [CL]",
    "BACA": "Balance cation-anion (BACA) [BACA]",

    # Autres indicateurs
    "FIRME": "Indice de fermeté / granulométrie [FIRME]",
    "NI_REF": "Indice azoté de référence [NI_REF]",
    "BE": "Besoins énergétiques [BE]",

    # Vitamines
    "VIT_A": "Vitamine A (UI/kg) [VIT_A]",
    "VIT_D": "Vitamine D (UI/kg) [VIT_D]",
    "VIT_E": "Vitamine E (mg/kg) [VIT_E]",

    # Acides aminés totaux (brut protéique BP)
    "LYS_BP": "Lysine totale (g/kg) [LYS_BP]",
    "HIS_BP": "Histidine totale (g/kg) [HIS_BP]",
    "ARG_BP": "Arginine totale (g/kg) [ARG_BP]",
    "THR_BP": "Thréonine totale (g/kg) [THR_BP]",
    "VAL_BP": "Valine totale (g/kg) [VAL_BP]",
    "MET_BP": "Méthionine totale (g/kg) [MET_BP]",
    "ILE_BP": "Isoleucine totale (g/kg) [ILE_BP]",
    "LEU_BP": "Leucine totale (g/kg) [LEU_BP]",
    "PHE_BP": "Phénylalanine totale (g/kg) [PHE_BP]",
    "ASP_BP": "Acide aspartique total (g/kg) [ASP_BP]",
    "SER_BP": "Sérine totale (g/kg) [SER_BP]",
    "GLU_BP": "Acide glutamique total (g/kg) [GLU_BP]",
    "PRO_BP": "Proline totale (g/kg) [PRO_BP]",
    "GLY_BP": "Glycine totale (g/kg) [GLY_BP]",
    "ALA_BP": "Alanine totale (g/kg) [ALA_BP]",
    "TYR_BP": "Tyrosine totale (g/kg) [TYR_BP]",
    "CYS_TRP_BP": "Cystéine + Tryptophane total (g/kg) [CYS_TRP_BP]",

    # Acides aminés digestibles iléaux (DI)
    "LYS_DI": "Lysine digestible iléale (g/kg MS) [LYS_DI]",
    "HIS_DI": "Histidine digestible iléale [HIS_DI]",
    "ARG_DI": "Arginine digestible iléale [ARG_DI]",
    "THR_DI": "Thréonine digestible iléale [THR_DI]",
    "VAL_DI": "Valine digestible iléale [VAL_DI]",
    "MET_DI": "Méthionine digestible iléale [MET_DI]",
    "ILE_DI": "Isoleucine digestible iléale [ILE_DI]",
    "LEU_DI": "Leucine digestible iléale [LEU_DI]",
    "PHE_DI": "Phénylalanine digestible iléale [PHE_DI]",
    "ASP_DI": "Acide aspartique digestible iléal [ASP_DI]",
    "SER_DI": "Sérine digestible iléale [SER_DI]",
    "GLU_DI": "Acide glutamique digestible iléal [GLU_DI]",
    "PRO_DI": "Proline digestible iléale [PRO_DI]",
    "GLY_DI": "Glycine digestible iléale [GLY_DI]",
    "ALA_DI": "Alanine digestible iléale [ALA_DI]",
    "TYR_DI": "Tyrosine digestible iléale [TYR_DI]",

    # Acides gras
    "C6_C10": "Acides gras C6 à C10 [C6_C10]",
    "C12_0": "Acide laurique (C12:0) [C12_0]",
    "C14_0": "Acide myristique (C14:0) [C14_0]",
    "C16_0": "Acide palmitique (C16:0) [C16_0]",
    "C16_1": "Acide palmitoléique (C16:1) [C16_1]",
    "C18_0": "Acide stéarique (C18:0) [C18_0]",
    "C18_1": "Acide oléique (C18:1) [C18_1]",
    "C18_2": "Acide linoléique (C18:2) [C18_2]",
    "C18_3": "Acide linolénique (C18:3) [C18_3]",
    "C20_0": "Acide arachidique (C20:0) [C20_0]",
    "C20_1": "Acide gadoléique (C20:1) [C20_1]",
    "C22_0": "Acide behénique (C22:0) [C22_0]",
    "C22_1": "Acide érucique (C22:1) [C22_1]",
    "C24_0": "Acide lignocérique (C24:0) [C24_0]",

    # Misc divers
    "IT3": "Indice IT3 (technologique) [IT3]",
    "PCT_UREE": "Pourcentage d’urée [PCT_UREE]",
    "IF": "Indice fourrager [IF]",
    "AGE": "Âge du fourrage/produit [AGE]",
    "CODE_FV": "Code filière végétale (INRA/AFZ) [CODE_FV]",
    "ALIM_ACHETE": "Aliment acheté (booléen) [ALIM_ACHETE]",
    "FC": "Facteur de correction [FC]",
    "PH": "pH du produit [PH]",
    "PAA": "Protéines véritables (AA) [PAA]",
}
df = df.rename(columns=column_mapping)
df.head()

,Code aliment INRA/AFZ [CODE],Nom de l'aliment (libellé officiel) [LIBELLE],Teneur en matière sèche (%) [TX_MS],Digestibilité de la matière organique (%) [DMO],UFL,UFV,"Matières azotées totales (Protéines brutes, %) [MAT]",Protéines digestibles dans l’intestin [PDI],Besoins protéiques ruminants (ancienne dénomination) [BPR],Teneur en amidon (%) [AMIDON],...,Leucine digestible iléale [LEU_DI],Phénylalanine digestible iléale [PHE_DI],Acide aspartique digestible iléal [ASP_DI],Sérine digestible iléale [SER_DI],Acide glutamique digestible iléal [GLU_DI],Proline digestible iléale [PRO_DI],Glycine digestible iléale [GLY_DI],Alanine digestible iléale [ALA_DI],Tyrosine digestible iléale [TYR_DI],Protéines véritables (AA) [PAA]
0,FV0010,"Fourrages verts, Prairies permanentes, Plaine ...",15.5,80.0,1.066528,1.051990,215.0,107.180561,59.383885,0.0,...,8.754891,5.324731,10.053103,4.719937,13.099694,4.727720,6.944416,6.870459,4.126154,0.896150
1,FV0160,"Fourrages verts, Prairies permanentes, Demi-mo...",16.7,79.0,1.051351,1.033720,166.0,96.297997,18.191062,0.0,...,8.672854,5.298694,10.186273,4.701315,13.113085,4.578761,6.926307,6.885575,4.136715,0.896150
2,FV2090,"Fourrages verts, Légumineuses fourragères, Luz...",14.4,77.0,1.012492,0.982038,246.0,103.498969,94.677659,0.0,...,8.567993,5.244497,12.360727,4.794089,12.384874,4.395962,6.684578,6.730078,4.077422,0.835556
3,FV0020,"Fourrages verts, Prairies permanentes, Plaine ...",16.6,77.0,1.019516,0.993305,172.0,95.325465,26.196524,0.0,...,8.683933,5.302210,10.168290,4.703830,13.111277,4.598877,6.928753,6.883534,4.135289,0.896150
4,FV0030,"Fourrages verts, Prairies permanentes, Plaine ...",17.2,72.0,0.941045,0.896853,133.0,85.207566,-2.849127,0.0,...,8.610466,5.223737,10.613906,4.707790,13.213988,4.385991,6.804251,6.865233,4.097362,0.896150


In [30]:
def search_data(query, df, col):
    """
    Recherche simple :
    - l'utilisateur tape une requête ("cereal", "wheat", etc.)
    - on cherche chaque mot dans une SEULE colonne
    - AND entre les mots
    """

    mots = query.lower().split()

    # Masque initial (tout vrai)
    mask = pd.Series([True] * len(df))

    for m in mots:
        mask = mask & df[col].astype(str).str.lower().str.contains(m, na=False)

    return df[mask]


# TEST
col = "Nom de l'aliment (libellé officiel) [LIBELLE]"
test_query = "blé tendre"
results = search_data(test_query, df, col)

print(f"Résultats pour la recherche : '{test_query}'")
print("Nombre de lignes trouvées :", len(results))
display(results.head())


Résultats pour la recherche : 'blé tendre'
Nombre de lignes trouvées : 8


,Code aliment INRA/AFZ [CODE],Nom de l'aliment (libellé officiel) [LIBELLE],Teneur en matière sèche (%) [TX_MS],Digestibilité de la matière organique (%) [DMO],UFL,UFV,"Matières azotées totales (Protéines brutes, %) [MAT]",Protéines digestibles dans l’intestin [PDI],Besoins protéiques ruminants (ancienne dénomination) [BPR],Teneur en amidon (%) [AMIDON],...,Leucine digestible iléale [LEU_DI],Phénylalanine digestible iléale [PHE_DI],Acide aspartique digestible iléal [ASP_DI],Sérine digestible iléale [SER_DI],Acide glutamique digestible iléal [GLU_DI],Proline digestible iléale [PRO_DI],Glycine digestible iléale [GLY_DI],Alanine digestible iléale [ALA_DI],Tyrosine digestible iléale [TYR_DI],Protéines véritables (AA) [PAA]
982,CC0150,"Céréales, Blé tendre",86.900002,87.0,1.19,1.21,126.0,89.0,-13.0,691.0,...,7.972080,5.011397,9.649692,4.729234,17.189439,5.482507,6.348880,6.411165,4.044119,1.0
983,CS0010,"Coproduits du blé, Son de blé tendre",86.900002,69.0,0.91,0.85,176.0,89.0,34.0,223.0,...,7.916443,4.881308,9.804482,4.726309,16.342181,5.106178,6.737842,6.472280,3.987210,1.0
988,CS0060,"Coproduits du blé, Farine basse de blé tendre",87.699997,90.0,1.31,1.35,152.0,94.0,6.0,695.0,...,7.956748,4.915043,9.783183,4.650540,16.842997,4.867013,6.687438,6.480462,4.028261,1.0
994,CS0120,"Coproduits du blé, Remoulage de blé tendre (tous)",87.900002,75.0,1.04,1.00,179.0,93.0,35.0,325.0,...,7.855501,4.835034,9.760652,5.466850,16.183753,5.051982,6.693965,6.450663,3.976721,1.0
996,CS0140,"Coproduits du blé, Remoulage de blé tendre, am...",88.099998,75.0,1.04,1.00,178.0,93.0,34.0,214.0,...,7.908161,4.882875,9.807373,4.729115,16.360309,5.130107,6.736748,6.471529,3.990609,1.0


### Appel à Ollama : résumé des résultats filtrés

Cette cellule met en place une fonction `ask_ollama()` qui permet 
d'appeler un modèle hébergé localement via Ollama.

Objectifs :
- envoyer un prompt (texte) au modèle
- récupérer sa réponse
- tester avec un résumé simple du sous-ensemble filtré

Ollama doit être lancé en arrière-plan (via `ollama serve`).


In [34]:
# Fonction pour interroger Ollama

def ask_ollama(prompt, model="llama3"):
    """
    Appelle Ollama et retourne la réponse complète.
    Gère le streaming JSON ligne par ligne (format Ollama).
    """
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt},
        stream=True
    )

    response.raise_for_status()

    full_text = ""

    # Lecture streaming (Ollama envoie plusieurs JSON consécutifs)
    for line in response.iter_lines():
        if not line:
            continue
        try:
            part = json.loads(line.decode("utf-8"))
            full_text += part.get("response", "")
        except Exception:
            # Ignore les fragments mal formés
            continue

    return full_text

# Test : Résumé intelligent des données filtrées

subset = results.head(15)
subset_md = subset.to_markdown(index=False)

prompt_test = f"""
Voici un tableau d'aliments pour l'élevage et leurs valeurs nutritionnelles :

{subset_md}

Tâche :
1. Décris en 4 à 6 phrases les différents type d'aliments visibles en prétant attention à la colonne "Nom de l'aliment (libellé officiel) [LIBELLE]".
2. Résume les tendances nutritionnelles globales de cet aliment.

Reste général et parle en français : ne cite pas de valeurs numériques exactes mais des plages de valeurs pour les composantes chimiques importantes.

Réponse :
"""

llm_output = ask_ollama(prompt_test, model="llama3")
print(llm_output)


Tâche :

1. Les différents types d'aliments présents dans ce tableau sont principalement des produits dérivés du blé, tels que le remoulage de blé tendre, divisé en plusieurs sous-groupes selon la quantité d'amidon contenu : < 30%, 30-40% et > 40%. Il y a également des aliments de base pour l'élevage, tels que les céréales et les protéines.

2. Les tendances nutritionnelles globales de cet aliment sont marquées par une importante quantité d'hydrogène soluble (amidon) qui représente la principale source d'énergie pour les animaux. Il y a également une présence modérée en protéines, azotés et minéraux tels que le calcium, le phosphore et le magnésium. Les taux de lipides sont généralement faibles. Les vitamines B et la fibre alimentaire ne sont pas spécifiquement mentionnées, mais il est probable qu'elles soient présentes dans des quantités modérées.


In [50]:
# Génération automatique de questions discriminantes

def generate_questions(df_subset, model="llama3"):

    subset_md = df_subset.to_markdown(index=False)

    prompt = f"""
Voici un extrait d'une base de données sous forme de tableau :

{subset_md}

Objectif :
Proposer 5 à 10 questions pertinentes qu'un utilisateur pourrait facilement comprendre, qui permettent de réduire l'incertitude
sur le profil nutritionnel d'un aliment.

Contraintes :
- Les questions doivent être courtes.
- Elles doivent permettre de comprendre les principales variables chimiques :
  protéines, fibres, énergie ...
- Ne fait pas de résumé : uniquement des questions.

Réponds en français, sous forme de liste numérotée.
"""

    return ask_ollama(prompt, model=model)


# Test de génération de questions
questions_output = generate_questions(subset, model="llama3")
print(questions_output)


Voici les 10 questions pertinentes pour comprendre le profil nutritionnel d'un aliment :

1. Quel est le taux de protéines dans cet aliments ?
2. Est-ce que l'aliment contient des fibres et si oui, en combien ?
3. Combien d'énergie (en kcal) offre cet aliment par portion ?
4. Le produit contient-il du sucre ? Si oui, combien ?
5. Quel est le ratio entre les glucides et les protéines dans cet aliments ?
6. Est-ce que l'aliment contient des graisses saturées ou insaturées ?
7. Combien de calories sont apportées par 100 grammes d'aliments ?
8. Y a-t-il des ingrédients qui pourraient causer une allergie ou une intolérance ?
9. Le produit est-il enrichi en vitamines ou minéraux particuliers (par exemple, calcium, fer) ?
10. Est-ce que l'aliment est considéré comme un aliments riche en glucides simples (sucre simple) ou complexes (fibres) ?


Sources : 

https://github.com/RamiKrispin/ollama-poc/blob/main/ollama-poc.ipynb

https://github.com/ollama/ollama